# 02 — Integración de dos modelos: $t_0=10$ s y $t_0=50$ s

Se integran dos simulaciones con la misma red `pynucastro` y la misma razón inicial $n/p=1/7$, pero con diferente tiempo inicial. Se guardan CSVs completos para la memoria y para las gráficas comparativas.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import scienceplots
    plt.style.use(["science", "grid"])
except Exception as exc:
    print("scienceplots no disponible, usando estilo por defecto:", exc)

from scipy.interpolate import interp1d
from scipy.integrate import solve_ivp
import importlib.util

BASE = Path.cwd()
FIG_DIR = BASE / "figures"
DATA_DIR = BASE / "data"
TAB_DIR = BASE / "tables"
for d in [FIG_DIR, DATA_DIR, TAB_DIR]:
    d.mkdir(exist_ok=True)

In [8]:
# Cargamos red generada en el notebook 01.
spec = importlib.util.spec_from_file_location("bbn_network", BASE / "bbn_network.py")
bbn = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bbn)

print("Núcleos:", bbn.names)
print("A:", bbn.A)
print("Z:", bbn.Z)

name_to_i = {str(name): i for i, name in enumerate(bbn.names)}
norm_to_real = {str(name).strip().lower(): str(name) for name in bbn.names}

# Compatibilidad entre nombres de pynucastro:
# algunas versiones escriben p,d,t; otras H1,H2,H3.
ALIASES = {
    "n":   ("n", "neutron"),
    "p":   ("p", "h1", "H1", "proton"),
    "d":   ("d", "h2", "H2"),
    "t":   ("t", "h3", "H3"),
    "he3": ("he3", "He3", "HE3"),
    "he4": ("he4", "He4", "HE4", "alpha"),
    "li7": ("li7", "Li7", "LI7"),
    "be7": ("be7", "Be7", "BE7"),
}

def nuc_name(key):
    for candidate in ALIASES[key]:
        candidate = str(candidate).strip()
        if candidate in name_to_i:
            return candidate
        cand_norm = candidate.lower()
        if cand_norm in norm_to_real:
            return norm_to_real[cand_norm]
    raise KeyError(
        f"No encuentro el núcleo '{key}'. Nombres disponibles en la red: {bbn.names}"
    )

def nuc_index(key):
    return name_to_i[nuc_name(key)]

def nuc_col(prefix, key):
    return f"{prefix}_{nuc_name(key).replace(' ', '')}"

CANON = {key: nuc_name(key) for key in ALIASES}
print("Aliases usados:", CANON)

Núcleos: ['n', 'H1', 'H2', 'H3', 'He3', 'He4', 'Li7', 'Be7']
A: [1 1 2 3 3 4 7 7]
Z: [0 1 1 1 2 2 3 4]
Aliases usados: {'n': 'n', 'p': 'H1', 'd': 'H2', 't': 'H3', 'he3': 'He3', 'he4': 'He4', 'li7': 'Li7', 'be7': 'Be7'}


In [9]:
# Trayectoria termodinámica forzada del notebook 00 / thermo_model.py.
# Esto debe coincidir con las figuras y los CSV termodinámicos de la memoria.

from thermo_model import make_thermo_functions, ETA10_PLANCK_APPROX

snapshots = pd.read_csv(DATA_DIR / "bbn_assignment_snapshots.csv")
thermo = make_thermo_functions(snapshots, eta10_ref=ETA10_PLANCK_APPROX)

T9_of_t = thermo.T9_of_t
T_of_t = thermo.T_of_t
rho_of_t = thermo.rho_of_t
eta_of_t = thermo.eta_of_t
n_gamma_cm3 = thermo.n_gamma_cm3

# Sanity check: debe reproducir los tres puntos del enunciado.
check = pd.DataFrame({
    "t_s": snapshots["t_s"],
    "T9_assignment": snapshots["T9"],
    "T9_model": T9_of_t(snapshots["t_s"].to_numpy(float)),
    "rho_assignment_g_cm3": snapshots["rho_g_cm3"],
    "rho_model_g_cm3": rho_of_t(snapshots["t_s"].to_numpy(float)),
    "eta10_effective": 1.0e10 * eta_of_t(snapshots["t_s"].to_numpy(float)),
})
check

,t_s,T9_assignment,T9_model,rho_assignment_g_cm3,rho_model_g_cm3,eta10_effective
0,50.0,1.7,1.7,0.000200,0.000200,12.088327
1,200.0,1.0,1.0,0.000020,0.000020,5.938995
2,1000.0,0.4,0.4,0.000002,0.000002,9.279679


In [10]:
# Funciones de integración y postprocesado.

def make_initial_Y(np_ratio=1.0/7.0):
    Y0 = np.zeros(bbn.nnuc)
    # np_ratio = n/p, con Y_n + Y_p = 1 para A=1.
    Yn = np_ratio / (1.0 + np_ratio)
    Yp = 1.0 / (1.0 + np_ratio)
    Y0[nuc_index("n")] = Yn
    Y0[nuc_index("p")] = Yp
    return Y0

def rhs_var(t, Y):
    Y_safe = np.maximum(np.asarray(Y, dtype=float), 0.0)
    return np.asarray(bbn.rhs(t, Y_safe, float(rho_of_t(t)), float(T_of_t(t))), dtype=float)

def run_model(t0, t_end=1.0e5, label="model", np_ratio=1.0/7.0):
    # Incluimos siempre los snapshots pedidos y un mallado logarítmico denso.
    requested = np.array([50.0, 200.0, 1000.0])
    requested = requested[(requested >= t0) & (requested <= t_end)]
    t_eval = np.unique(np.concatenate([np.geomspace(t0, t_end, 700), requested]))
    Y0 = make_initial_Y(np_ratio=np_ratio)
    sol = solve_ivp(
        rhs_var, (t0, t_end), Y0,
        method="LSODA", t_eval=t_eval,
        rtol=1e-6, atol=1e-30, max_step=2.0,
    )
    if not sol.success:
        raise RuntimeError(f"Falló {label}: {sol.message}")
    return sol

def solution_to_dataframe(sol, model_label, t0, np_ratio):
    Y_hist = np.maximum(sol.y.T, 0.0)
    X_hist = Y_hist * bbn.A[np.newaxis, :]
    df = pd.DataFrame({
        "model": model_label,
        "t0_s": t0,
        "n_over_p_initial": np_ratio,
        "p_over_n_initial": 1.0 / np_ratio,
        "t_s": sol.t,
        "T9": T9_of_t(sol.t),
        "T_K": T_of_t(sol.t),
        "rho_g_cm3": rho_of_t(sol.t),
        "eta": eta_of_t(sol.t),
        "eta10": 1.0e10 * eta_of_t(sol.t),
        "baryon_sum": np.sum(X_hist, axis=1),
    })
    for i, name in enumerate(bbn.names):
        clean = str(name).replace(" ", "")
        df[f"Y_{clean}"] = Y_hist[:, i]
        df[f"X_{clean}"] = X_hist[:, i]
    return df

def abundance_ratios(df):
    out = df.copy()
    Yp = out[nuc_col("Y", "p")].replace(0, np.nan)

    def ratio(key):
        col = nuc_col("Y", key)
        return out[col] / Yp if col in out.columns else np.nan

    out["D/H"] = ratio("d")
    out["T/H"] = ratio("t")
    out["He3/H"] = ratio("he3")
    out["Li7/H"] = ratio("li7")
    out["Be7/H"] = ratio("be7")
    out["Li7_plus_Be7_over_H"] = out["Li7/H"].fillna(0) + out["Be7/H"].fillna(0)
    out["X_He4"] = out[nuc_col("X", "he4")]
    out["X_H1"] = out[nuc_col("X", "p")]
    return out

def snapshot_table(df, snapshot_times=(50.0, 200.0, 1000.0)):
    rows = []
    for t in snapshot_times:
        if t < df["t_s"].min() or t > df["t_s"].max():
            continue
        row = {"model": df["model"].iloc[0], "t0_s": df["t0_s"].iloc[0], "t_s": t}
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            row[col] = float(np.interp(t, df["t_s"], df[col]))
        rows.append(row)
    return pd.DataFrame(rows)

In [11]:
# Ejecutamos las dos simulaciones.
# Puedes cambiar el n/p inicial de cada caso aquí si quieres hacer tests.
MODEL_CONFIGS = [
    {"label": "t0_10s", "t0": 10.0, "n_over_p": 1.0/5.0},
    {"label": "t0_50s", "t0": 50.0, "n_over_p": 1.0/7.0},
]

models = {}
for cfg in MODEL_CONFIGS:
    sol = run_model(cfg["t0"], label=cfg["label"], np_ratio=cfg["n_over_p"])
    df = abundance_ratios(solution_to_dataframe(sol, cfg["label"], cfg["t0"], cfg["n_over_p"]))
    models[cfg["label"]] = df
    df.to_csv(DATA_DIR / f"bbn_evolution_{cfg['label']}.csv", index=False)

model10 = models["t0_10s"]
model50 = models["t0_50s"]

all_models = pd.concat([model10, model50], ignore_index=True)
all_models.to_csv(DATA_DIR / "bbn_evolution_all_models.csv", index=False)

snap_all = pd.concat([snapshot_table(model10), snapshot_table(model50)], ignore_index=True)
snap_all.to_csv(DATA_DIR / "bbn_snapshots_all_models.csv", index=False)

# Finales al final de la integración
final_all = all_models.sort_values("t_s").groupby("model", as_index=False).tail(1)
final_all.to_csv(DATA_DIR / "bbn_final_abundances_all_models.csv", index=False)

snap_all[["model", "t_s", "T9", "rho_g_cm3", "eta10", "D/H", "He3/H", "X_He4", "Li7/H", "Be7/H", "Li7_plus_Be7_over_H", "baryon_sum"]]

 lsoda--  warning..internal t (=r1) and h (=r2) are  
       such that in the machine, t + h = t on the next step  
       (h = step size). solver will continue anyway  
      in above,  r1 =  0.1000000000000D+02   r2 =  0.7914439001993D-28
 lsoda--  warning..internal t (=r1) and h (=r2) are  
       such that in the machine, t + h = t on the next step  
       (h = step size). solver will continue anyway  
      in above,  r1 =  0.1000000000000D+02   r2 =  0.7914439001993D-28
 lsoda--  warning..internal t (=r1) and h (=r2) are  
       such that in the machine, t + h = t on the next step  
       (h = step size). solver will continue anyway  
      in above,  r1 =  0.1000000000000D+02   r2 =  0.7914439001993D-24
 lsoda--  warning..internal t (=r1) and h (=r2) are  
       such that in the machine, t + h = t on the next step  
       (h = step size). solver will continue anyway  
      in above,  r1 =  0.1000000000000D+02   r2 =  0.7914439001993D-24
 lsoda--  warning..internal t (=r1) 

,model,t_s,T9,rho_g_cm3,eta10,D/H,He3/H,X_He4,Li7/H,Be7/H,Li7_plus_Be7_over_H,baryon_sum
0,t0_10s,50.0,1.7,0.000200,12.088327,3.044123e-19,4.480068e-06,3.333120e-01,3.040262e-21,1.740871e-06,1.740871e-06,1.0
1,t0_10s,200.0,1.0,0.000020,5.938995,3.025164e-19,4.271261e-06,3.333119e-01,8.055620e-21,1.851602e-06,1.851602e-06,1.0
2,t0_10s,1000.0,0.4,0.000002,9.279679,3.559219e-19,4.259598e-06,3.333119e-01,8.632026e-20,1.858433e-06,1.858433e-06,1.0
3,t0_50s,50.0,1.7,0.000200,12.088327,1.142857e-27,2.402034e-55,1.614236e-84,0.000000e+00,0.000000e+00,0.000000e+00,1.0
4,t0_50s,200.0,1.0,0.000020,5.938995,2.725777e-08,5.280194e-06,2.498880e-01,4.406025e-16,8.823541e-08,8.823541e-08,1.0
5,t0_50s,1000.0,0.4,0.000002,9.279679,5.587486e-09,5.262808e-06,2.498880e-01,4.705110e-17,9.465002e-08,9.465002e-08,1.0


In [12]:
# Tabla resumen final con sólo los observables más relevantes.
obs_cols = ["model", "t0_s", "t_s", "T9", "rho_g_cm3", "eta10", "D/H", "He3/H", "X_He4", "Li7/H", "Be7/H", "Li7_plus_Be7_over_H", "baryon_sum"]
final_compact = final_all[obs_cols].copy()
final_compact.to_csv(DATA_DIR / "bbn_final_compact.csv", index=False)
final_compact

,model,t0_s,t_s,T9,rho_g_cm3,eta10,D/H,He3/H,X_He4,Li7/H,Be7/H,Li7_plus_Be7_over_H,baryon_sum
1404,t0_50s,50.0,100000.0,0.001015,3.955179e-07,1.122689e+08,4.968346e-09,0.000005,0.249888,1.292026e-16,9.469180e-08,9.469180e-08,1.0
702,t0_10s,10.0,100000.0,0.001015,3.955179e-07,1.122689e+08,3.969620e-19,0.000004,0.333312,1.203562e-16,1.858478e-06,1.858478e-06,1.0
